In [3]:
import cv2 #Used for computer vision tasks like image processing and video analysis.
import os  #provides functions for interacting with the operating system, such as file and directory handling.

# Load the Haar cascade for eye detection
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml') 

# Create a directory to store the cropped eye images
output_dir = 'cropped_eyes1'
os.makedirs(output_dir, exist_ok=True)

# Load the video
video_path = 'video01.mp4' 
cap = cv2.VideoCapture(video_path)

frame_number = 0 # Initializes a counter to keep track of the number of frames processed.
eye_count = 0 #Initializes a counter to keep track of the number of eyes detected and saved.
  
while cap.isOpened():# Loop through the video frames
    ret, frame = cap.read()# Read a frame from the video
    if not ret:# If there are no more frames, exit the loop
        break

    # Convert the frame to grayscale for eye detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect eyes in the frame
    eyes = eye_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Save each detected eye as a cropped image
    for (x, y, w, h) in eyes:
        eye_img = frame[y:y+h, x:x+w] #Crops the region of the frame containing the detected eye.
        eye_filename = os.path.join(output_dir, f'eye_{eye_count}.png')
        cv2.imwrite(eye_filename, eye_img)
        eye_count += 1 #Increments the eye_count counter by 1 after each eye is saved.

    frame_number += 1 # Increments the frame_number counter by 1 after each frame is processed.

cap.release() #Releases the video file, closing it and freeing up resources.
cv2.destroyAllWindows() #Closes any OpenCV windows that were opened during the processing.

print(f"Processed {frame_number} frames and saved {eye_count} eye images to '{output_dir}'")


Processed 364 frames and saved 901 eye images to 'cropped_eyes1'


In [4]:
def number_folders(dataset_path):
    folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    folder_number_mapping = {}
    
    for i, folder_name in enumerate(folders):
        folder_number_mapping[folder_name] = i
    
    return folder_number_mapping

# Path to your dataset directory
dataset_path = 'video01image'  
folder_mapping = number_folders(dataset_path) #Calls the number_folders function with the dataset_path argument. 

# Print the mapping of folders to numbers
for folder, number in folder_mapping.items():
    print(f"Folder '{folder}' is assigned number {number}")


Folder 'down' is assigned number 0
Folder 'left' is assigned number 1
Folder 'right' is assigned number 2
Folder 'up' is assigned number 3


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import tensorflow as tf#It is used for building and training deep learning models.
from tensorflow.keras import layers, models
import joblib


In [6]:
#Function to Load Images and Labels from a Folder
def load_images_from_folder(folder):
    images = []
    labels = []
    #Loop through the Folders and Load Images
    for label in os.listdir(folder):
        label_path = os.path.join(folder, label)
        if os.path.isdir(label_path):
            #Load Each Image File
            for filename in os.listdir(label_path):
                img_path = os.path.join(label_path, filename)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    img = cv2.resize(img, (50, 50))  # Resize to 50x50 pixels
                    images.append(img) #Adds the processed image to the images list.
                    labels.append(label)#Adds the corresponding label (the folder name) to the labels list.
    return np.array(images), np.array(labels)

# Load dataset
dataset_path = 'video01image' 
X, y = load_images_from_folder(dataset_path)

# Normalize images
X = X / 255.0
X_flattened = X.reshape(X.shape[0], -1)  # Flatten images for traditional ML models

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Save the classes to a file
np.save('classes.npy', label_encoder.classes_)

# Split data into training and testing sets
X_train_flat, X_test_flat, y_train, y_test = train_test_split(X_flattened, y, test_size=0.2, random_state=42)


In [7]:
# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_flat, y_train)
lr_predictions = lr_model.predict(X_test_flat)
lr_accuracy = accuracy_score(y_test, lr_predictions)
print(f"Logistic Regression Accuracy: {lr_accuracy * 1.00:.2f}%")


Logistic Regression Accuracy: 1.00%


In [8]:
# Support Vector Machine
svm_model = SVC(kernel='linear')
svm_model.fit(X_train_flat, y_train)
svm_predictions = svm_model.predict(X_test_flat)
svm_accuracy = accuracy_score(y_test, svm_predictions)
print(f"SVM Accuracy: {svm_accuracy * 1.00:.2f}%")


SVM Accuracy: 1.00%


In [9]:
# Reshape for CNN
X_train_cnn = X.reshape(-1, 50, 50, 1)
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(X_train_cnn, y, test_size=0.2, random_state=42)

# CNN model
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(50, 50, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(label_encoder.classes_), activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

cnn_model.fit(X_train_cnn, y_train_cnn, epochs=10, batch_size=32, validation_split=0.2)

cnn_loss, cnn_accuracy = cnn_model.evaluate(X_test_cnn, y_test_cnn)
print(f"CNN Accuracy: {cnn_accuracy * 1.00:.2f}%")


Epoch 1/10


c:\Users\manya\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.3405 - loss: 1.3354 - val_accuracy: 0.5818 - val_loss: 1.1836
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6717 - loss: 1.1189 - val_accuracy: 0.9273 - val_loss: 0.8448
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9109 - loss: 0.7053 - val_accuracy: 0.9818 - val_loss: 0.3236
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9951 - loss: 0.2381 - val_accuracy: 1.0000 - val_loss: 0.0701
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9974 - loss: 0.0644 - val_accuracy: 1.0000 - val_loss: 0.0173
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 0.0141 - val_accuracy: 1.0000 - val_loss: 0.0071
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.0046 - val_accuracy: 1.0000 - val_loss: 0.0047
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 1.0000 - val_loss: 0.0027
Epoch 9/10


In [10]:
# Determine the best model and save it
best_model = None
best_accuracy = 0
best_model_name = ""
#Checking Logistic Regression Model
if lr_accuracy > best_accuracy:
    best_accuracy = lr_accuracy
    best_model = lr_model
    best_model_name = "Logistic Regression"

if svm_accuracy > best_accuracy:
    best_accuracy = svm_accuracy
    best_model = svm_model
    best_model_name = "SVM"

if cnn_accuracy > best_accuracy:
    best_accuracy = cnn_accuracy
    best_model = cnn_model
    best_model_name = "CNN"
#Saving the Best Model
if best_model_name != "CNN":
    # Save traditional ML models using joblib
    joblib.dump(best_model, 'best_eye_movement_model.pkl')
else:
    # Save CNN model using TensorFlow's built-in save method
    cnn_model.save('best_eye_movement_cnn_model.h5')

print(f"The best model is {best_model_name} with an accuracy of {best_accuracy * 1.00:.2f}%")


The best model is Logistic Regression with an accuracy of 1.00%


In [11]:
import pyautogui #For controlling the mouse cursor based on eye movements. 


In [12]:
# Load the model
model_path = 'best_eye_movement_model.pkl'  # For traditional ML models
cnn_model_path = 'best_eye_movement_cnn_model.h5'  # For CNN models

try:
    # Try loading traditional ML model first
    model = joblib.load(model_path)
    model_type = 'traditional_ml'
except:
    # If traditional ML model is not found, load the CNN model
    model = tf.keras.models.load_model(cnn_model_path)
    model_type = 'cnn'

# Load the label encoder
label_encoder = LabelEncoder()
label_encoder.classes_ = np.load('classes.npy')  
# Define a function to move the cursor based on predicted direction
def move_cursor(prediction):
    x, y = pyautogui.position()
    if prediction == 1:  # Example condition, modify based on your model's output
        pyautogui.moveTo(x - 20, y)  # Move cursor to the left (or corresponding action)
    elif prediction == 2:
        pyautogui.moveTo(x + 20, y)  # Move cursor to the right (or corresponding action)
    elif prediction == 3:
        pyautogui.moveTo(x, y - 20)  # Move cursor upward (or corresponding action)
    elif prediction == 0:
        pyautogui.moveTo(x, y + 20)  # Move cursor downward (or corresponding action)

# Load the video
video_path = 'video01.mp4'  
cap = cv2.VideoCapture(video_path)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert to grayscale for eye detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
    eyes = eye_cascade.detectMultiScale(gray, 1.1, 4)

    for (x, y, w, h) in eyes:
        # Crop the eye region
        eye = gray[y:y+h, x:x+w]
        eye_resized = cv2.resize(eye, (50, 50))

        # Prepare the eye image for prediction
        if model_type == 'traditional_ml':
            eye_flattened = eye_resized.flatten() / 255.0
            eye_flattened = eye_flattened.reshape(1, -1)
            prediction = model.predict(eye_flattened)
        else:
            eye_normalized = eye_resized / 255.0
            eye_normalized = eye_normalized.reshape(1, 50, 50, 1)
            prediction = np.argmax(model.predict(eye_normalized), axis=1)

        # Perform the action based on the prediction (without explicitly mentioning directions)
        move_cursor(prediction)

    # Show the video frame with eye detection
    cv2.imshow('Eye Movement Detection', frame)

    # Press 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture and close all windows
cap.release()
cv2.destroyAllWindows()
